<a href="https://colab.research.google.com/github/v-krishna07/polyglots-shorthand-sentiment/blob/main/inter_iit_csai_ps_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset,concatenate_datasets

tweets = load_dataset("Abhishek4896/hindi-english-code-mixed-tweets-sentiment", split="train")
youtube = load_dataset("shae2977/hinglish-youtube-sentiments-dataset", split="train")

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}

def normalize_tweets(example):
    return {"text": example["tweet"], "label": LABEL2ID[example["sentiment"].strip().lower()]}

def normalize_youtube(example):
    return {"text": example["comment"], "label": LABEL2ID[example["sentiment"].strip().lower()]}

tweets_clean = tweets.map(normalize_tweets, remove_columns=tweets.column_names)
youtube_clean = youtube.map(normalize_youtube, remove_columns=youtube.column_names)

fused = concatenate_datasets([tweets_clean, youtube_clean]).shuffle(seed=42)
print(fused)

In [ ]:
import emoji
from collections import Counter
import re

# Strip: skin-tone modifiers
SKIN_TONE_MODIFIERS = "".join(chr(c) for c in range(0x1F3FB, 0x1F400))  # U+1F3FB–1F3FF
STRIP_PATTERN = re.compile(f"[\uFE0F\u200D{SKIN_TONE_MODIFIERS}]")
VARIATION_SELECTOR = "\uFE0F"

def clean_text(text):
    return text.replace(VARIATION_SELECTOR, "")
def normalize_emoji(e):
    return STRIP_PATTERN.sub("", e)

emoji_counter = Counter()
for text in fused["text"]:
    for found in emoji.emoji_list(text):
        e = normalize_emoji(clean_text(found["emoji"]))
        if e:  # guard against empty string if normalization strips everything
            emoji_counter[e] += 1

top_emojis = [e for e, _ in emoji_counter.most_common(50)]
print(len(top_emojis), top_emojis)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME="l3cube-pune/hing-roberta"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(top_emojis)

model= AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=3)
model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
device

print(f"we have vocab size: {len(tokenizer)}")
print(f"Embedding maxtrix shape: {model.get_input_embeddings().weight.shape}")


In [ ]:
import random
import math
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from datasets import concatenate_datasets

print("Splitting...")
alL_labels=np.array(fused["label"])
splitter=StratifiedShuffleSplit(n_splits=1,test_size=0.1,random_state=42)
train_idx,val_idx=next(splitter.split(np.zeros(len(alL_labels)),alL_labels))
train_raw=fused.select(train_idx)
val_raw=fused.select(val_idx)
print(f"Split complete! Train: {len(train_raw)} rows | Val: {len(val_raw)} rows\n")

random.seed(42)
VOWELS=set("aeiouAEIOU")
QWERTY={
    'q':'wa', 'w':'qes', 'e':'wrd', 'r':'etf', 't':'rgy', 'y':'thu', 'u':'yji', 'i':'uko', 'o':'ipl', 'p':'ol',
    'a':'qsz', 's':'awdz', 'd':'sefx', 'f':'drgc', 'g':'ftyv', 'h':'gybn', 'j':'huki', 'k':'jilm', 'l':'kop',
    'z':'asx', 'x':'zsdc', 'c':'xdfv', 'v':'cfgb', 'b':'vghn', 'n':'bhjm', 'm':'njk',
}

def drop_vowels(word):
  if len(word)<=2: return word
  min_length=math.ceil(len(word)*6)
  vowel_index=[i for i in range(1,len(word)) if word[i] in VOWELS]
  max_droppable=len(word)-min_length
  if max_droppable<=0 or not vowel_index: return word
  drop = set(random.sample(vowel_index,min(max_droppable,len(vowel_index))))
  return "".join(c for i,c in enumerate(word) if i not in drop)

def elongate(word):
  if not word:return word
  inde = random.randint(0,len(word)-1)
  return word[:inde]+word[inde]*random.randint(2,4)+word[inde+1:]

def qwerty_swap(word):
  if not word:return word
  inde = random.randint(0,len(word)-1)
  ch = word[inde].lower()
  if ch in QWERTY:
    return word[:inde]+random.choice(QWERTY[ch])+word[inde+1:]
  return word

CORRUPTIONS=[drop_vowels,elongate,qwerty_swap]

def inject_noise(example, max_corruptions=2):
    words = str(example["text"]).split()
    eligible = [i for i, w in enumerate(words) if len(w) > 2]
    targets = set(random.sample(eligible, min(max_corruptions, len(eligible)))) if eligible else set()
    noisy = [random.choice(CORRUPTIONS)(w) if i in targets else w for i, w in enumerate(words)]
    return {"text": " ".join(noisy), "label": example["label"]}

print("Injecting phonetic noise to build robustness...")
noisy_train = train_raw.map(inject_noise)
train_raw_augmented = concatenate_datasets([train_raw, noisy_train]).shuffle(seed=42)

VARIATION_SELECTOR = "\uFE0F"
def clean_text(text):
    return str(text).replace(VARIATION_SELECTOR, "")

def tokenize_fn(batch):
    # Apply clean_text to every string in the batch BEFORE passing to tokenizer
    cleaned_texts = [clean_text(text) for text in batch["text"]]
    # Using 128 max_length to prepare for strict IO Binding latency lock later
    return tokenizer(cleaned_texts, truncation=True, max_length=128)

print("Tokenizing datasets and cleaning emoji variation selectors...")
train_enc = train_raw_augmented.map(tokenize_fn, batched=True, remove_columns=["text"]).rename_column("label", "labels")
val_enc = val_raw.map(tokenize_fn, batched=True, remove_columns=["text"]).rename_column("label", "labels")

train_enc.set_format("torch")
val_enc.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_enc, batch_size=16, shuffle=True, collate_fn=collator, pin_memory=True)
val_loader = DataLoader(val_enc, batch_size=32, shuffle=False, collate_fn=collator, pin_memory=True)

print(f"Original Train: {len(train_raw)} | Augmented Train: {len(train_raw_augmented)}")
print(f"Data Prep Complete! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
import numpy as np
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, accuracy_score
from transformers import get_cosine_schedule_with_warmup

def train_sentiment_model(dataset_train, dataset_val, epochs=10):
    model.to(device)

    labels = np.array(dataset_train["label"])
    counts = np.bincount(labels, minlength=3)
    weights = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float32).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

    def tok_fn(batch):
        cleaned_texts = [clean_text(text) for text in batch["text"]]
        return tokenizer(cleaned_texts, truncation=True, max_length=128)

    train_ds = dataset_train.map(tok_fn, batched=True, remove_columns=["text"]).rename_column("label", "labels")
    val_ds = dataset_val.map(tok_fn, batched=True, remove_columns=["text"]).rename_column("label", "labels")
    train_ds.set_format("torch"); val_ds.set_format("torch")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collator, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collator, pin_memory=True)

    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grouped_params = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]

    optimizer = AdamW(grouped_params, lr=2e-5)
    steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*steps), num_training_steps=steps)
    scaler = GradScaler("cuda")

    best_acc, patience = 0.0, 0
    print(f"Starting training on {len(dataset_train)} rows...")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with autocast('cuda'):
                loss = loss_fn(model(**batch).logits, batch.pop("labels"))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item()

        # Eval
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for b in val_loader:
                b = {k: v.to(device, non_blocking=True) for k, v in b.items()}
                with autocast('cuda'):
                    preds = torch.argmax(model(**b).logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(b.pop("labels").cpu().numpy())

        acc = accuracy_score(all_true, all_preds)
        val_f1 = f1_score(all_true, all_preds, average="macro")
        print(f"Epoch {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | Accuracy: {acc*100:.2f}% | F1: {val_f1:.4f}")

        if acc > best_acc:
            best_acc, patience = acc, 0
            model.save_pretrained("./hinglish_model_checkpoint")
            tokenizer.save_pretrained("./hinglish_model_checkpoint")
        else:
            patience += 1
            if patience >= 5:
                print("Early stopping triggered.")
                break
    return best_acc

print("\n--- PHASE 1: Training on Augmented Data ---")
train_sentiment_model(train_raw_augmented, val_raw, epochs=20)

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoModelForSequenceClassification

print("Downloading unlabeled dataset...")
unlabeled_ds = load_dataset("Yugrathee28/Hinglish-dataset", split="train")
original_texts = unlabeled_ds["text"]

def tokenize_unl(batch):
  cleaned_texts = [clean_text(text) for text in batch["text"]]
  return tokenizer(cleaned_texts, truncation=True, max_length=128)

print("Preparing unlabeled data for inference...")
tok_unl = unlabeled_ds.map(tokenize_unl, batched=True)
tok_unl.set_format(type="torch", columns=["input_ids", "attention_mask"])
unl_loader = DataLoader(tok_unl, batch_size=64, collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))

pseudo_texts, pseudo_labels, current_idx = [], [], 0

# Reload the best Phase 1 model
print("Loading best Phase 1 model...")
model = AutoModelForSequenceClassification.from_pretrained("./hinglish_model_checkpoint").to(device)
model.eval()

print("Extracting >95% confidence labels...")
with torch.inference_mode():
    for batch in tqdm(unl_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with autocast('cuda'):
            logits = model(**batch).logits
        max_probs, preds = torch.max(F.softmax(logits, dim=-1), dim=-1)

        for i in range(len(preds)):
            if max_probs[i] > 0.95:
                pseudo_texts.append(original_texts[current_idx + i])
                pseudo_labels.append(preds[i].item())
        current_idx += len(preds)


pseudo_ds = Dataset.from_dict({"text": pseudo_texts, "label": pseudo_labels})
train_massive = concatenate_datasets([train_raw_augmented, pseudo_ds]).shuffle(seed=42)
print(f"Massive Dataset Built: {len(train_massive)} rows (Original: {len(train_raw_augmented)} | Pseudo: {len(pseudo_ds)})")

print("\n--- PHASE 2: Training on Larger Data ---")
train_sentiment_model(train_massive, val_raw, epochs=20)








In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTOptimizer
from optimum.onnxruntime.configuration import AutoOptimizationConfig
from transformers import AutoTokenizer
import json
import os

pytorch_dir = "./hinglish_model_checkpoint"
onnx_dir = "./hinglish_onnx_model"
fp16_dir = "./hinglish_onnx_fp16"

os.makedirs(fp16_dir, exist_ok=True)

# Permanently fix the Mistral regex bug before export
# open add close
with open(f"{pytorch_dir}/tokenizer_config.json", "r") as f:
    cfg = json.load(f)
cfg["fix_mistral_regex"] = True
with open(f"{pytorch_dir}/tokenizer_config.json", "w") as f:
    json.dump(cfg, f)


print("Step 1: Exporting PyTorch to base ONNX graph...")
tokenizer = AutoTokenizer.from_pretrained(pytorch_dir)
base_onnx = ORTModelForSequenceClassification.from_pretrained(pytorch_dir, export=True)
base_onnx.save_pretrained(onnx_dir)

print("Step 2: compiling FP16 weights (Level O4)...")
optimizer = ORTOptimizer.from_pretrained(onnx_dir)
opt_config = AutoOptimizationConfig.O4()
optimizer.optimize(save_dir=fp16_dir, optimization_config=opt_config)

tokenizer.save_pretrained(fp16_dir)
print(f"\nSuccess! FP16 optimized model saved to {fp16_dir}")

In [ ]:
# Benchmark test
import time
import torch
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification

# this is to force CUDA
try:
    ort.preload_dlls()
except AttributeError:
    pass

fp16_dir = "./hinglish_onnx_fp16"

print(f"Active Providers: {ort.get_available_providers()}")
assert "CUDAExecutionProvider" in ort.get_available_providers(), "GPU not bound! Make sure you restarted the session."

print("Loading FP16 ONNX model with CUDA and IO Binding...")
tokenizer = AutoTokenizer.from_pretrained(fp16_dir)

onnx_model = ORTModelForSequenceClassification.from_pretrained(
    fp16_dir,
    file_name="model_optimized.onnx",
    provider="CUDAExecutionProvider",
    use_io_binding=True
)

# EXACT 128-token padding for memory reuse
inputs = tokenizer("bhai ye toh ekdum mast hai 🤡", return_tensors="pt", padding="max_length", max_length=128)
inputs = {k: v.to("cuda") for k, v in inputs.items()}

print("Warming up NVIDIA Tensor Cores...")
for _ in range(30):
    _ = onnx_model(**inputs)
torch.cuda.synchronize()

print("Benchmarking strict FP16 GPU latency (500 passes)...")
latencies = []
for _ in range(500):
    start = time.perf_counter()
    _ = onnx_model(**inputs)
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)

print("── FINAL LATENCY BENCHMARK ──")
print(f"Average Latency: {np.mean(latencies):.2f} ms")
print(f"P99 Latency:     {np.percentile(latencies, 99):.2f} ms")
print(f"P75 Latency:     {np.percentile(latencies, 75):.2f} ms")
print(f"P50 Latency:     {np.percentile(latencies, 50):.2f} ms")

Failed to load libcufft.so.12: libcufft.so.12: cannot open shared object file: No such file or directory
Please follow https://onnxruntime.ai/docs/install/#cuda-and-cudnn to install CUDA.
Active Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Loading FP16 ONNX model with CUDA and IO Binding...
Warming up NVIDIA Tensor Cores...
Benchmarking strict FP16 GPU latency (200 passes)...

── FINAL PRODUCTION LATENCY BENCHMARK ──
Average Latency: 3.32 ms
P99 Latency:     8.24 ms
P75 Latency:     3.95 ms
P50 Latency:     2.58 ms


In [ ]:
!pip install onnxruntime
import onnxruntime as ort
print("Available Providers:", ort.get_available_providers())

In [ ]:
# Sample testing code

import time
import torch
import numpy as np
import onnxruntime
import torch.nn.functional as F
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification

# Force CUDA library binding
try:
    onnxruntime.preload_dlls()
except AttributeError:
    pass

onnx_dir = "./hinglish_onnx_fp16"
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("Loading optimized FP16 ONNX Engine...")
tokenizer = AutoTokenizer.from_pretrained(onnx_dir, fix_mistral_regex=True)
engine = ORTModelForSequenceClassification.from_pretrained(
    onnx_dir,
    file_name="model_optimized.onnx",
    provider="CUDAExecutionProvider",
    use_io_binding=True
)

test_samples = [
    "bhai sach bata raha hu, product ekdum fire hai 🔥 maza aa gaya!",
    "Paisa barbaad ho gaya yaar, bilkul ghatiya customer support hai 😡 scam pura",
    "Kal match kitne baje start hoga? Mujhe time verify karna tha bas.",
    "Bhai kya solid update diya hai team ne, dil jeet liya ❤️",
    "kuch samjh nhi aa rha bakwas app h dlt kr rha hu",
]

print("\n── REAL-WORLD HINGLISH INFERENCE (ONNX) ──")

# Warmup pass (Tensor Cores need a few passes to reach peak speed)
warmup_input = tokenizer("warmup", return_tensors="pt", padding="max_length", max_length=128)
warmup_input = {k: v.to("cuda") for k, v in warmup_input.items()}
for _ in range(5):
    _ = engine(**warmup_input)
torch.cuda.synchronize()

for text in test_samples:
    # 1. Clean variation selectors (the bug we fixed)
    clean_input = str(text).replace("\uFE0F", "")

    # 2. Strict 128 padding for stable IO-Binding memory allocation
    inputs = tokenizer(clean_input, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # 3. Time the raw ONNX execution
    start = time.perf_counter()
    with torch.no_grad():
        logits = engine(**inputs).logits
    torch.cuda.synchronize()
    latency = (time.perf_counter() - start) * 1000

    # 4. Process outputs
    probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    pred_idx = probs.argmax()

    print(f"\nInput:      {text}")
    print(f"Sentiment:  {label_map[pred_idx]} ({probs[pred_idx]*100:.1f}%)")
    print(f"Latency:    {latency:.2f} ms")

Failed to load libcufft.so.12: libcufft.so.12: cannot open shared object file: No such file or directory
Please follow https://onnxruntime.ai/docs/install/#cuda-and-cudnn to install CUDA.
Loading optimized FP16 ONNX Engine...

── REAL-WORLD HINGLISH INFERENCE (ONNX) ──

Input:      bhai sach bata raha hu, product ekdum fire hai 🔥 maza aa gaya!
Sentiment:  Positive (93.8%)
Latency:    4.55 ms

Input:      Paisa barbaad ho gaya yaar, bilkul ghatiya customer support hai 😡 scam pura
Sentiment:  Negative (91.3%)
Latency:    4.47 ms

Input:      Kal match kitne baje start hoga? Mujhe time verify karna tha bas.
Sentiment:  Neutral (96.6%)
Latency:    4.50 ms

Input:      Bhai kya solid update diya hai team ne, dil jeet liya ❤️
Sentiment:  Positive (94.2%)
Latency:    4.54 ms

Input:      kuch samjh nhi aa rha bakwas app h dlt kr rha hu
Sentiment:  Negative (91.5%)
Latency:    4.43 ms
